# Hi AI Clip Worker (Kaggle GPU)

Reads `job.json`, writes short clips to `/kaggle/working/clips/`, writes `manifest.json`, then exits so GPU quota stops.

Kernel id: `godswillchiemeziem/hi-ai-clip-worker`

In [1]:
import json, os, time, pathlib, subprocess

WORK = pathlib.Path('/kaggle/working')
CLIPS = WORK / 'clips'
CLIPS.mkdir(parents=True, exist_ok=True)

def find_job():
    p = WORK / 'job.json'
    if p.exists():
        return json.loads(p.read_text())
    root = pathlib.Path('/kaggle/input')
    if root.exists():
        for f in root.rglob('job.json'):
            return json.loads(f.read_text())
    # default dry job so Save & Run All still produces outputs
    return {
        'job_id': 'dry-' + str(int(time.time())),
        'channel': 'test',
        'clip_seconds': 5,
        'num_clips': 2,
        'prompts': [
            'cinematic vertical shot soft light, no text',
            'calm atmosphere vertical 9:16, no text'
        ]
    }

job = find_job()
print('JOB', job.get('job_id'), 'clips', job.get('num_clips'))
(WORK / 'job_echo.json').write_text(json.dumps(job, indent=2))


JOB dry-1792852751 clips 2


208

In [2]:
# Placeholder clips (solid color). Replace later with LTX/Wan GPU models.
sec = float(job.get('clip_seconds') or 5)
n = int(job.get('num_clips') or 2)
prompts = job.get('prompts') or ['placeholder']
manifest = []

for i in range(n):
    out = CLIPS / f'clip_{i:02d}.mp4'
    color = ['black', 'gray', 'navy', 'maroon'][i % 4]
    cmd = [
        'ffmpeg', '-y', '-f', 'lavfi', '-i', f'color=c={color}:s=1080x1920:d={sec}',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-t', str(sec), str(out)
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        ok = out.exists() and out.stat().st_size > 1000
    except Exception as e:
        print('ffmpeg fail', e)
        ok = False
    manifest.append({
        'index': i,
        'path': out.name,
        'prompt': prompts[i % len(prompts)],
        'ok': ok,
        'seconds': sec
    })
    print('clip', i, ok, out)

(WORK / 'manifest.json').write_text(json.dumps({
    'job_id': job.get('job_id'),
    'channel': job.get('channel'),
    'clips': manifest,
    'status': 'complete'
}, indent=2))
print('DONE — kernel will exit and free GPU')


clip 0 True /kaggle/working/clips/clip_00.mp4
clip 1 True /kaggle/working/clips/clip_01.mp4
DONE — kernel will exit and free GPU
